# BirdCLEF 2026 — Training Soundscapes → Training Data

Converts competition training soundscapes into `.npy` mel spectrograms using ground truth labels.

**Approach:**
- Uses `start` and `end` timestamps from `train_soundscapes_labels.csv` — no model inference needed
- Splits composite labels like `a;b;c` into separate rows
- Output: folder of `.npy` files + a manifest CSV with `filename` and `primary_label`
- Plugs directly into `BirdDataset` alongside existing training data

**Directory structure (relative to this notebook in `Code/`):**
```
birdclef-2026/
├── train_soundscapes/              ← input .ogg files
├── train_soundscapes_labels.csv    ← input labels
├── Pseudo_training_data/           ← output .npy files + manifest (created by this notebook)
└── Code/
    └── soundscape-to-training-data.ipynb  ← this notebook
```

In [1]:
%pip install librosa kagglehub tqdm


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# =========================
# 1. IMPORTS
# =========================
import os
import numpy as np
import pandas as pd
import librosa
import kagglehub
from pathlib import Path
from tqdm.auto import tqdm

print('Libraries loaded successfully')

Libraries loaded successfully


In [3]:
# =========================
# 2. CONFIGURATION
# =========================
class CFG:
    # Audio — must match training notebook exactly
    SR = 32000
    DURATION = 5
    CHUNK_SIZE = SR * DURATION

    # Mel spectrogram — must match training notebook exactly
    N_MELS = 128
    FMAX = 16000

    # Paths — relative to this notebook sitting inside Code/
    BASE_DIR        = Path('../')
    SOUNDSCAPES_DIR = BASE_DIR / 'train_soundscapes'
    SOUNDSCAPE_LABELS_CSV      = BASE_DIR / 'train_soundscapes_labels.csv'
    TRAIN_CSV       = BASE_DIR / 'train.csv'
    OUTPUT_DIR      = BASE_DIR / 'Pseudo_training_data'
    MANIFEST_PATH   = OUTPUT_DIR / 'soundscape_manifest.csv'

    # Kaggle upload config — update these before running Cell 8
    KAGGLE_HANDLE   = 'gany24558/birdclef-groundtruth-soundscape-composite-pl'  # username/dataset-slug

CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Base dir            : {CFG.BASE_DIR.resolve()}')
print(f'Soundscapes dir     : {CFG.SOUNDSCAPES_DIR.resolve()}')
print(f'Soundscapes exists  : {CFG.SOUNDSCAPES_DIR.exists()}')
print(f'SOUNDSCAPE Labels CSV          : {CFG.SOUNDSCAPE_LABELS_CSV.resolve()}')
print(f'SOUNDSCAPE Labels CSV exists   : {CFG.SOUNDSCAPE_LABELS_CSV.exists()}')
print(f'Output dir          : {CFG.OUTPUT_DIR.resolve()}')

Base dir            : /Users/ganapathychidambaram/Desktop/birds/birdclef-2026
Soundscapes dir     : /Users/ganapathychidambaram/Desktop/birds/birdclef-2026/train_soundscapes
Soundscapes exists  : True
SOUNDSCAPE Labels CSV          : /Users/ganapathychidambaram/Desktop/birds/birdclef-2026/train_soundscapes_labels.csv
SOUNDSCAPE Labels CSV exists   : True
Output dir          : /Users/ganapathychidambaram/Desktop/birds/birdclef-2026/Pseudo_training_data


In [4]:
import pandas as pd
from pathlib import Path

taxonomy_df = pd.read_csv('../taxonomy.csv')
print("List of Taxonomy Columns: ", taxonomy_df.columns.tolist())
print("First 5 rows of Taxonomy: ", taxonomy_df.head(5))
print(taxonomy_df['class_name'].value_counts())

List of Taxonomy Columns:  ['primary_label', 'inat_taxon_id', 'scientific_name', 'common_name', 'class_name']
First 5 rows of Taxonomy:    primary_label  inat_taxon_id         scientific_name  \
0       1161364        1161364            Guyalna cuta   
1        116570         116570           Caiman yacare   
2       1176823        1176823  Leptodactylus luctator   
3       1491113        1491113       Adenomera guarani   
4       1595929        1595929       Lysapsus limellum   

                  common_name class_name  
0                Guyalna cuta    Insecta  
1  Southern Spectacled Caiman   Reptilia  
2               Wrestler Frog   Amphibia  
3    Guaraní leaf-litter frog   Amphibia  
4      Uruguay Harlequin Frog   Amphibia  
class_name
Aves        162
Amphibia     35
Insecta      28
Mammalia      8
Reptilia      1
Name: count, dtype: int64


#### Verify if the unknown labels from the soundscape CSV are in taxonomy:

In [5]:
static_unknown_labels = ['22961', '23158', '24321', '517063', '65380']  # from earlier
taxonomy_df['primary_label'] = taxonomy_df['primary_label'].astype(str)
for label in static_unknown_labels:
    match = taxonomy_df[taxonomy_df['primary_label'] == label]
    if len(match) > 0:
        print(f"{label} → {match['common_name'].values[0]} ({match['class_name'].values[0]})")
    else:
        print(f"{label} → NOT FOUND in taxonomy")

22961 → Pointedbelly Frog (Amphibia)
23158 → Pale-legged Weeping Frog (Amphibia)
24321 → Mato Grosso Snouted Tree Frog (Amphibia)
517063 → Southern Orange-legged Leaf Frog (Amphibia)
65380 → Dwarf Tree Frog (Amphibia)


### Check which 28 are missing

In [6]:
# =========================
# 3. LOAD & INSPECT LABELS
# =========================
labels_df = pd.read_csv(CFG.SOUNDSCAPE_LABELS_CSV)
print(f'Total rows in Soundscape labels CSV : {len(labels_df)}')
print(f'Columns           : {list(labels_df.columns)}')
print()
print(labels_df.head(10))

# Count how many rows have composite labels
composite = labels_df['primary_label'].str.contains(';', na=False).sum()
print(f'\nRows with composite labels (a;b;c): {composite}')

# =========================
# VERIFY labels map to known species in train.csv
# =========================
train_df = pd.read_csv(CFG.TRAIN_CSV)
species_in_train = set(train_df['primary_label'].unique())

# Collect all unique labels from soundscape CSV
all_soundscape_labels = set()
for label_str in labels_df['primary_label'].dropna():
    for label in str(label_str).split(';'):
        all_soundscape_labels.add(label.strip())

soundscape_only_labels = all_soundscape_labels - species_in_train
print(f'\nTotal unique labels in soundscape CSV : {len(all_soundscape_labels)}')
print(f'Number of Soundscape Labels found in train.csv             : {len(all_soundscape_labels - soundscape_only_labels)}')
# isn't above the length of 'species in train'?
print(f'Number of Soundscape Labels NOT in train.csv (will skip)   : {len(soundscape_only_labels)}')
print(f'TBD: Why did we decide to skip?  ')
if soundscape_only_labels:
    print(f'Soundscape Labels NOT in train.csv: {sorted(soundscape_only_labels)[:10]}')

Total rows in Soundscape labels CSV : 1478
Columns           : ['filename', 'start', 'end', 'primary_label']

                                    filename     start       end  \
0  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:00  00:00:05   
1  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:05  00:00:10   
2  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:10  00:00:15   
3  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:15  00:00:20   
4  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:20  00:00:25   
5  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:25  00:00:30   
6  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:30  00:00:35   
7  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:35  00:00:40   
8  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:40  00:00:45   
9  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:45  00:00:50   

                    primary_label  
0  22961;23158;24321;517063;65380  
1  22961;23158;24321;517063;65380  
2  22961;23158;24321;517063;65380

### Label cross verification


In [7]:
print(f"Species in train.csv    : {train_df['primary_label'].nunique()}")
print(f"Species in taxonomy.csv : {len(taxonomy_df)}")
print(f"Missing from training   : {len(taxonomy_df) - train_df['primary_label'].nunique()}")

Species in train.csv    : 206
Species in taxonomy.csv : 234
Missing from training   : 28


In [8]:
train_species = set(train_df['primary_label'].astype(str).unique())
taxonomy_species = set(taxonomy_df['primary_label'].astype(str).unique())

missing_in_train = taxonomy_species - train_species
print(f'Missing in train.csv species ({len(missing_in_train)}):')
for label in sorted(missing_in_train):
    match = taxonomy_df[taxonomy_df['primary_label'].astype(str) == label]
    print(f"  {label} → {match['common_name'].values[0]} ({match['class_name'].values[0]})")

Missing in train.csv species (28):
  1491113 → Guaraní leaf-litter frog (Amphibia)
  25073 → Chiasmocleis mehelyi (Amphibia)
  47158son01 → Insect sonotype01 (Insecta)
  47158son02 → Insect sonotype02 (Insecta)
  47158son03 → Insect sonotype03 (Insecta)
  47158son04 → Insect sonotype04 (Insecta)
  47158son05 → Insect sonotype05 (Insecta)
  47158son06 → Insect sonotype06 (Insecta)
  47158son07 → Insect sonotype07 (Insecta)
  47158son08 → Insect sonotype08 (Insecta)
  47158son09 → Insect sonotype09 (Insecta)
  47158son10 → Insect sonotype10 (Insecta)
  47158son11 → Insect sonotype11 (Insecta)
  47158son12 → Insect sonotype12 (Insecta)
  47158son13 → Insect sonotype13 (Insecta)
  47158son14 → Insect sonotype14 (Insecta)
  47158son15 → Insect sonotype15 (Insecta)
  47158son16 → Insect sonotype16 (Insecta)
  47158son17 → Insect sonotype17 (Insecta)
  47158son18 → Insect sonotype18 (Insecta)
  47158son19 → Insect sonotype19 (Insecta)
  47158son20 → Insect sonotype20 (Insecta)
  47158son21 → 

In [9]:
# =========================
# 4. HELPER FUNCTIONS
# =========================
def audio_to_melspec(audio_chunk):
    """Convert audio array to mel spectrogram dB.
    Parameters must match BirdDataset preprocessing exactly."""
    S = librosa.feature.melspectrogram(
        y=audio_chunk,
        sr=CFG.SR,
        n_mels=CFG.N_MELS,
        fmax=CFG.FMAX
    )
    return librosa.power_to_db(S, ref=np.max)


def pad_or_trim(audio, target_length):
    """Ensure clip is exactly target_length samples.
    Pads with zeros if shorter, trims if longer."""
    if len(audio) < target_length:
        return np.pad(audio, (0, target_length - len(audio)))
    return audio[:target_length]


def time_to_samples(time_str, sr):
    """Convert HH:MM:SS timestamp to sample index.
    e.g. '00:00:05' -> 160000 (at SR=32000)"""
    parts = str(time_str).strip().split(':')
    hours, minutes, seconds = int(parts[0]), int(parts[1]), int(parts[2])
    total_seconds = hours * 3600 + minutes * 60 + seconds
    return int(total_seconds * sr)

In [10]:
# ==========================================
# DEDUPLICATE SOUNDSCAPE LABELS
# ==========================================

import pandas as pd

print("Before deduplication:")
print(f"Rows: {len(labels_df)}")

# ------------------------------------------
# Normalize composite labels
# ------------------------------------------

def normalize_labels(label_str):
    labels = [
        l.strip()
        for l in str(label_str).split(';')
        if l.strip() and l.strip() != 'nan'
    ]
    # Remove duplicates + sort
    labels = sorted(list(set(labels)))
    return ';'.join(labels)

# Normalize labels
labels_df['primary_label'] = (
    labels_df['primary_label']
    .astype(str)
    .apply(normalize_labels)
)

# ------------------------------------------
# Remove duplicate rows
# ------------------------------------------
labels_df = labels_df.drop_duplicates(
    subset=[
        'filename',
        'start',
        'end',
        'primary_label'
    ]
).reset_index(drop=True)

print("\nAfter deduplication:")
print(f"Rows: {len(labels_df)}")

print("\nSample rows:")
print(labels_df.head())

Before deduplication:
Rows: 1478

After deduplication:
Rows: 739

Sample rows:
                                    filename     start       end  \
0  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:00  00:00:05   
1  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:05  00:00:10   
2  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:10  00:00:15   
3  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:15  00:00:20   
4  BC2026_Train_0039_S22_20211231_201500.ogg  00:00:20  00:00:25   

                    primary_label  
0  22961;23158;24321;517063;65380  
1  22961;23158;24321;517063;65380  
2  22961;23158;24321;517063;65380  
3  22961;23158;24321;517063;65380  
4  22961;23158;24321;517063;65380  


In [11]:
import json
# =========================
# 5. MAIN PROCESSING LOOP
# =========================
manifest_rows = []
errors = []

# Group by filename — loads each .ogg only once even if it has multiple rows
grouped = labels_df.groupby('filename')
print(f'Unique soundscape files to process: {len(grouped)}')

for filename, group in tqdm(grouped, desc='Processing soundscapes'):
    file_path = CFG.SOUNDSCAPES_DIR / filename

    if not file_path.exists():
        errors.append(f'File not found: {file_path}')
        continue

    # Load full audio file once per soundscape
    try:
        y, _ = librosa.load(file_path, sr=CFG.SR)
    except Exception as e:
        errors.append(f'Load error {filename}: {e}')
        continue

    for _, row in group.iterrows():
        start_sample = time_to_samples(row['start'], CFG.SR)
        end_sample   = time_to_samples(row['end'],   CFG.SR)

        # Extract clip between start and end timestamps
        clip = y[start_sample:end_sample]

        # Pad or trim to exactly 5 seconds
        clip = pad_or_trim(clip, CFG.CHUNK_SIZE)

        # Convert to mel spectrogram
        melspec = audio_to_melspec(clip)

        # Split composite labels e.g. 'a;b;c' -> ['a', 'b', 'c']
        labels = [l.strip() for l in str(row['primary_label']).split(';')]

        # Save one .npy per clip — store just the filename (not full path)
        # This makes the manifest portable when uploaded to Kaggle
        start_str    = str(row['start']).replace(':', '-')
        end_str      = str(row['end']).replace(':', '-')
        clip_id      = f"{filename.replace('.ogg', '')}__s{start_str}_e{end_str}"
        npy_filename = f"{clip_id}.npy"
        npy_path     = CFG.OUTPUT_DIR / npy_filename
        np.save(npy_path, melspec)

        """ 
        # Create one manifest row per label
        # Store just the filename — training notebook will prepend the correct base path
        for label in labels:
            if not label or label == 'nan':
                continue
            manifest_rows.append({
                'filename'      : npy_filename,   # just filename, not full path
                'primary_label' : label
            })
        """ 

        if len(labels)==0:
            continue
        manifest_rows.append({
            'filename'      : npy_filename,   # just filename, not full path
            'primary_label' : json.dumps(labels)
        })

print(f'\n✅ Done!')
print(f'Total manifest rows (one per clip) : {len(manifest_rows)}')
print(f'Errors encountered                  : {len(errors)}')
if errors:
    for e in errors[:5]:
        print(f'  {e}')

Unique soundscape files to process: 66


Processing soundscapes:   0%|          | 0/66 [00:00<?, ?it/s]


✅ Done!
Total manifest rows (one per clip) : 739
Errors encountered                  : 0


In [12]:
import json

# =========================
# 6. SAVE MANIFEST
# =========================

manifest_df = pd.DataFrame(manifest_rows)

manifest_df.to_csv(CFG.MANIFEST_PATH, index=False)

print(f'Manifest saved to   : {CFG.MANIFEST_PATH.resolve()}')
print(f'Total rows          : {len(manifest_df)}')
print(f'Unique .npy files   : {manifest_df["filename"].nunique()}')

# ==========================================
# EXPLODE MULTILABEL SPECIES FOR ANALYSIS
# ==========================================

all_species = []

for labels_json in manifest_df['primary_label']:

    try:
        labels = json.loads(labels_json)

        if not isinstance(labels, list):
            labels = [labels]

        all_species.extend(labels)

    except Exception:
        continue

species_counts = pd.Series(all_species).value_counts()

print(f'Unique species      : {len(species_counts)}')

print()
print('Sample rows:')
print(manifest_df.head(10))

print()
print('Most common species (top 10):')
print(species_counts.head(10))

print()
print('Least common species (bottom 10):')
print(species_counts.tail(10))

Manifest saved to   : /Users/ganapathychidambaram/Desktop/birds/birdclef-2026/Pseudo_training_data/soundscape_manifest.csv
Total rows          : 739
Unique .npy files   : 739
Unique species      : 75

Sample rows:
                                            filename  \
0  BC2026_Train_0001_S08_20250606_030007__s00-00-...   
1  BC2026_Train_0001_S08_20250606_030007__s00-00-...   
2  BC2026_Train_0001_S08_20250606_030007__s00-00-...   
3  BC2026_Train_0001_S08_20250606_030007__s00-00-...   
4  BC2026_Train_0001_S08_20250606_030007__s00-00-...   
5  BC2026_Train_0001_S08_20250606_030007__s00-00-...   
6  BC2026_Train_0001_S08_20250606_030007__s00-00-...   
7  BC2026_Train_0001_S08_20250606_030007__s00-00-...   
8  BC2026_Train_0001_S08_20250606_030007__s00-00-...   
9  BC2026_Train_0001_S08_20250606_030007__s00-00-...   

                                       primary_label  
0  ["47158son13", "47158son17", "47158son22", "47...  
1  ["47158son13", "47158son17", "47158son21", "47...  
2  [

In [13]:
# =========================
# 7. SANITY CHECK
# =========================
sample_filename = manifest_df['filename'].iloc[0]
sample = np.load(CFG.OUTPUT_DIR / sample_filename)

print(f'Sample .npy shape  : {sample.shape}')   # Expect (128, 313)
print(f'Value range        : {sample.min():.2f} to {sample.max():.2f}')  # Expect ~-80 to 0 dB
print()
print('✅ Sanity check passed!')

# =========================
# 8. File Numbers check
# =========================

import pandas as pd
from pathlib import Path

output_dir = Path('../Pseudo_training_data')
npy_files = list(output_dir.glob('*.npy'))
manifest_df = pd.read_csv(output_dir / 'soundscape_manifest.csv')

print(f'.npy files on disk      : {len(npy_files)}')
print(f'Unique files in manifest: {manifest_df["filename"].nunique()}')
print(f'Total manifest rows     : {len(manifest_df)}')

Sample .npy shape  : (128, 313)
Value range        : -80.00 to 0.00

✅ Sanity check passed!
.npy files on disk      : 739
Unique files in manifest: 739
Total manifest rows     : 739


In [14]:
import os
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_76c34848cd62be4b168fbbc1eaa44ada'

In [15]:
# =========================
# 8. UPLOAD TO KAGGLE
# =========================
# Requires kagglehub and valid Kaggle API credentials (~/.kaggle/kaggle.json)
# Update KAGGLE_HANDLE in CFG before running

print(f'Uploading {CFG.OUTPUT_DIR.resolve()} to Kaggle...')
print(f'Dataset handle: {CFG.KAGGLE_HANDLE}')
print()

kagglehub.dataset_upload(
    CFG.KAGGLE_HANDLE,
    str(CFG.OUTPUT_DIR.resolve()),
    version_notes='Ground truth soundscape features — 739 clips, 6244 manifest rows'
)

print()
print('✅ Upload complete!')
print(f'Find your dataset at: https://www.kaggle.com/datasets/{CFG.KAGGLE_HANDLE}')
print()
print('Next steps:')
print('1. Attach this dataset to your training notebook on Kaggle')
print('2. Load the manifest and prepend the Kaggle input path:')
print('   soundscape_df = pd.read_csv("/kaggle/input/birdclef-soundscape-features/soundscape_manifest.csv")')
print('   soundscape_df["filename"] = "/kaggle/input/birdclef-soundscape-features/" + soundscape_df["filename"]')
print('3. Concatenate with train_df and pass into BirdDataset')

Uploading /Users/ganapathychidambaram/Desktop/birds/birdclef-2026/Pseudo_training_data to Kaggle...
Dataset handle: gany24558/birdclef-groundtruth-soundscape-composite-pl

Uploading Dataset https://kaggle.com/datasets/gany24558/birdclef-groundtruth-soundscape-composite-pl ...
More than 50 files detected, creating a zip archive...
Starting upload for file /var/folders/q7/bbr773zn4y1025z46ls_d5_w0000gn/T/tmpd1kewkxq/archive.zip


Uploading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 119M/119M [01:30<00:00, 1.31MB/s]

Upload successful: /var/folders/q7/bbr773zn4y1025z46ls_d5_w0000gn/T/tmpd1kewkxq/archive.zip (113MB)


Your dataset has been created.
Files are being processed...
See at: https://kaggle.com/datasets/gany24558/birdclef-groundtruth-soundscape-composite-pl

✅ Upload complete!
Find your dataset at: https://www.kaggle.com/datasets/gany24558/birdclef-groundtruth-soundscape-composite-pl

Next steps:
1. Attach this dataset to your training notebook on Kaggle
2. Load the manifest and prepend the Kaggle input path:
   soundscape_df = pd.read_csv("/kaggle/input/birdclef-soundscape-features/soundscape_manifest.csv")
   soundscape_df["filename"] = "/kaggle/input/birdclef-soundscape-features/" + soundscape_df["filename"]
3. Concatenate with train_df and pass into BirdDataset
